# OpenAQ Data Crawling — 6 Hanoi Stations

Crawl hourly PM2.5, CO, NO2, O3, SO2 for 6 stations in Hanoi from the OpenAQ v3 API.  
Stations are read from `data/raw/DataAOD/Hanoi/Stations.xlsx`.  
Output CSVs are saved to the same folder.

## 1. Imports & Configuration

In [1]:
import os
import time
import random
from pathlib import Path

import pandas as pd
import requests

# API
OPENAQ_BASE_URL = "https://api.openaq.org/v3"
OPENAQ_API_KEY  = "8e8c3a875320048ecc88c70fcba8a72b10466adada7fb420c851038a59f3b987"
HEADERS         = {"X-API-Key": OPENAQ_API_KEY}

# Paths
PROJECT_ROOT  = Path("D:/Bussiness_plan/Multimodal_PM25")
STATIONS_FILE = PROJECT_ROOT / "data/raw/DataAOD/Hanoi/Stations.xlsx"
OUTPUT_DIR    = PROJECT_ROOT / "data/raw/DataAOD/Hanoi"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Crawl settings
GLOBAL_START_DATE = "2021-01-01"
GLOBAL_END_DATE   = "2026-05-15"
LIMIT             = 1000
WINDOW_DAYS       = 14     # 14-day windows to avoid 408 timeouts

TARGET_PARAMETERS = {
    "co":   "CO mass µg/m³",
    "no2":  "NO₂ mass µg/m³",
    "o3":   "O₃ mass µg/m³",
    "pm25": "PM2.5 µg/m³",
    "so2":  "SO₂ mass µg/m³",
}

print("Config loaded.")
print(f"  Output dir : {OUTPUT_DIR}")
print(f"  Date range : {GLOBAL_START_DATE}  →  {GLOBAL_END_DATE}")


Config loaded.
  Output dir : D:\Bussiness_plan\Multimodal_PM25\data\raw\DataAOD\Hanoi
  Date range : 2021-01-01  →  2026-05-15


## 2. Helper Functions

In [2]:
def request_json(url, params=None, timeout=60, max_retries=8):
    last_exc = None
    for attempt in range(max_retries):
        try:
            resp = requests.get(url, headers=HEADERS, params=params, timeout=timeout)

            if resp.status_code == 429:
                raw_reset = resp.headers.get("x-ratelimit-reset", "")
                try:
                    reset_val = float(raw_reset)
                    wait_s = max(reset_val - time.time(), 1) if reset_val > 1e9 else max(reset_val, 5)
                except (ValueError, TypeError):
                    wait_s = 60
                sleep_s = wait_s + random.uniform(2, 5)
                print(f"  429 rate-limit → sleep {sleep_s:.1f}s (attempt {attempt+1}/{max_retries})")
                time.sleep(sleep_s)
                continue

            if resp.status_code in {408, 500, 502, 503, 504}:
                sleep_s = min(120, 2 ** attempt + random.uniform(1, 3))
                print(f"  {resp.status_code} server error → retry in {sleep_s:.1f}s")
                time.sleep(sleep_s)
                continue

            resp.raise_for_status()
            return resp.json()

        except requests.RequestException as e:
            last_exc = e
            if attempt == max_retries - 1:
                raise
            sleep_s = min(120, 2 ** attempt + random.uniform(1, 3))
            print(f"  Network error: {e} → retry in {sleep_s:.1f}s")
            time.sleep(sleep_s)

    raise RuntimeError(
        f"Exhausted {max_retries} retries for {url}. "
        f"Last error: {last_exc}. "
        "Likely persistent 429/408 — wait a few minutes then re-run."
    )


def get_location_metadata(location_id: int) -> dict:
    payload = request_json(f"{OPENAQ_BASE_URL}/locations/{location_id}")
    results = payload.get("results", [])
    if not results:
        raise ValueError(f"Location {location_id} not found.")
    loc    = results[0]
    coords = loc.get("coordinates") or {}
    return {
        "location_id":   loc.get("id"),
        "location_name": loc.get("name"),
        "timezone":      loc.get("timezone"),
        "latitude":      coords.get("latitude"),
        "longitude":     coords.get("longitude"),
    }


def get_location_sensors(location_id: int) -> list:
    all_rows, page = [], 1
    while True:
        payload = request_json(
            f"{OPENAQ_BASE_URL}/locations/{location_id}/sensors",
            params={"limit": 1000, "page": page},
        )
        rows  = payload.get("results", [])
        found = int((payload.get("meta") or {}).get("found") or 0)
        all_rows.extend(rows)
        if not rows or page * 1000 >= found:
            break
        page += 1
    return all_rows


def build_sensor_inventory(location_id: int) -> pd.DataFrame:
    sensors = get_location_sensors(location_id)
    rows = []
    for s in sensors:
        p     = s.get("parameter") or {}
        pname = str(p.get("name", "")).lower()
        units = p.get("units")
        if pname not in TARGET_PARAMETERS or units != "µg/m³":
            continue
        dt_first = (s.get("datetimeFirst") or {}).get("utc")
        dt_last  = (s.get("datetimeLast")  or {}).get("utc")
        if not dt_first or not dt_last:
            continue
        rows.append({
            "sensor_id":         s.get("id"),
            "parameter_name":    pname,
            "datetimeFirst_utc": dt_first,
            "datetimeLast_utc":  dt_last,
        })
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(["parameter_name", "sensor_id"]).reset_index(drop=True)
    return df


def split_windows(start_ts, end_ts, window_days=14):
    start_ts = pd.Timestamp(start_ts)
    end_ts   = pd.Timestamp(end_ts)
    windows, cur = [], start_ts
    step = pd.Timedelta(days=window_days)
    while cur <= end_ts:
        nxt = min(cur + step, end_ts)
        windows.append((cur, nxt))
        cur = nxt + pd.Timedelta(seconds=1)
    return windows


def fetch_sensor_hours(sensor_id: int, start_ts, end_ts) -> list:
    start_str = pd.Timestamp(start_ts).strftime("%Y-%m-%dT%H:%M:%SZ")
    end_str   = pd.Timestamp(end_ts).strftime("%Y-%m-%dT%H:%M:%SZ")
    all_results, page = [], 1
    while True:
        payload = request_json(
            f"{OPENAQ_BASE_URL}/sensors/{sensor_id}/hours",
            params={"datetime_from": start_str, "datetime_to": end_str,
                    "limit": LIMIT, "page": page},
        )
        results = payload.get("results", [])
        if not results:
            break
        all_results.extend(results)
        found = int((payload.get("meta") or {}).get("found") or 0)
        if page * LIMIT >= found:
            break
        page += 1
    return all_results


def extract_rows(results, parameter_name: str) -> list:
    rows = []
    for r in results:
        dt_to   = (r.get("period") or {}).get("datetimeTo") or {}
        summary = r.get("summary") or {}
        rows.append({
            "datetime_utc":   dt_to.get("utc"),
            "parameter_name": parameter_name,
            "value":          summary.get("avg"),
        })
    return rows


print("All helpers defined.")


All helpers defined.


## 3. Load Stations

In [3]:
df_raw = pd.read_excel(STATIONS_FILE)
df_raw.columns = [str(c).strip() for c in df_raw.columns]

df_stations = pd.DataFrame({
    "Location_id":   df_raw["Location"].astype(int),
    "Location_name": df_raw["Name"].str.strip(),
    "Start Date":    pd.Timestamp(GLOBAL_START_DATE),
    "End Date":      pd.Timestamp(GLOBAL_END_DATE),
})

print(f"Loaded {len(df_stations)} stations:")
display(df_stations)


Loaded 34 stations:


,Location_id,Location_name,Start Date,End Date
0,18,SPARTAN - Vietnam Acad. Sci.,2021-01-01,2026-05-15
1,2539,US Diplomatic Post: Hanoi,2021-01-01,2026-05-15
2,7441,Hanoi,2021-01-01,2026-05-15
3,307169,nồng độ pm,2021-01-01,2026-05-15
4,1285357,SPARTAN - Vietnam Acad. Sci.,2021-01-01,2026-05-15
5,2161290,An Khánh,2021-01-01,2026-05-15
6,2161291,Cầu Diễn,2021-01-01,2026-05-15
7,2161292,"Số 46, phố Lưu Quang Vũ",2021-01-01,2026-05-15
8,2161293,Chúc Sơn,2021-01-01,2026-05-15
9,2161294,Cung thiếu nhi,2021-01-01,2026-05-15


## 4. Crawl All Stations

In [4]:
all_station_dfs = []

for _, station_row in df_stations.iterrows():
    loc_id   = int(station_row["Location_id"])
    loc_name = str(station_row["Location_name"]).strip()
    start_ts = pd.Timestamp(station_row["Start Date"]).tz_localize("UTC")
    end_ts   = pd.Timestamp(station_row["End Date"]).tz_localize("UTC")

    print(f"\n{'='*60}")
    print(f"Station {loc_id}: {loc_name}")
    print(f"  Range: {start_ts.date()}  →  {end_ts.date()}")

    # Metadata + sensor inventory
    try:
        meta      = get_location_metadata(loc_id)
        inventory = build_sensor_inventory(loc_id)
    except Exception as e:
        print(f"  [SKIP] metadata/sensor error: {e}")
        continue

    if inventory.empty:
        print("  [SKIP] No matching sensors found.")
        continue

    print(f"  Sensors: {len(inventory)}")

    # Fetch hourly data per sensor
    all_rows = []

    for s_row in inventory.itertuples(index=False):
        eff_start = max(start_ts, pd.Timestamp(s_row.datetimeFirst_utc).tz_convert("UTC"))
        eff_end   = min(end_ts,   pd.Timestamp(s_row.datetimeLast_utc).tz_convert("UTC"))
        if eff_start > eff_end:
            continue

        windows = split_windows(eff_start, eff_end, window_days=WINDOW_DAYS)
        print(f"  {s_row.parameter_name} | sensor={s_row.sensor_id} | {len(windows)} windows")

        for i, (w_start, w_end) in enumerate(windows, start=1):
            print(f"    [{i:03d}/{len(windows)}] {w_start.date()} → {w_end.date()}", end="  ")
            try:
                results = fetch_sensor_hours(s_row.sensor_id, w_start, w_end)
                rows    = extract_rows(results, s_row.parameter_name)
                all_rows.extend(rows)
                print(f"({len(rows)} records)")
            except Exception as e:
                print(f"[ERROR: {e}] — skipped")
            time.sleep(random.uniform(0.8, 1.5))

    if not all_rows:
        print("  [SKIP] No data fetched.")
        continue

    # Aggregate + pivot wide
    df_r = pd.DataFrame(all_rows)
    df_r["datetime_utc"]   = pd.to_datetime(df_r["datetime_utc"], utc=True, errors="coerce")
    df_r["parameter_name"] = df_r["parameter_name"].str.lower()
    df_r = df_r.dropna(subset=["datetime_utc"])

    df_agg = df_r.groupby(["datetime_utc", "parameter_name"], as_index=False)["value"].mean()

    final_df = (
        df_agg
        .pivot(index="datetime_utc", columns="parameter_name", values="value")
        .reset_index()
        .rename(columns=TARGET_PARAMETERS)
    )

    final_df["location_id"]   = meta["location_id"]
    final_df["location_name"] = meta["location_name"]
    final_df["latitude"]      = meta["latitude"]
    final_df["longitude"]     = meta["longitude"]

    col_order = [
        "datetime_utc", "location_id", "location_name", "latitude", "longitude",
        "CO mass µg/m³", "NO₂ mass µg/m³", "O₃ mass µg/m³", "PM2.5 µg/m³", "SO₂ mass µg/m³",
    ]
    for c in col_order:
        if c not in final_df.columns:
            final_df[c] = pd.NA
    final_df = final_df[col_order].sort_values("datetime_utc").reset_index(drop=True)

    # Save per-station CSV
    safe = loc_name.replace("/", "_").replace("\\", "_").replace(",", "").replace(".", "").strip()
    out_csv = OUTPUT_DIR / f"{loc_id}_{safe}.csv"
    final_df.to_csv(out_csv, index=False, encoding="utf-8-sig")
    print(f"  Saved: {out_csv.name}  shape={final_df.shape}")
    print(f"  PM2.5 rows: {final_df['PM2.5 µg/m³'].notna().sum()}")

    all_station_dfs.append(final_df)
    time.sleep(3)

# Combine all stations
print(f"\n{'='*60}")
if all_station_dfs:
    df_all = (
        pd.concat(all_station_dfs, ignore_index=True)
        .sort_values(["location_id", "datetime_utc"])
        .reset_index(drop=True)
    )
    out_all = OUTPUT_DIR / "all_stations_crawled.csv"
    df_all.to_csv(out_all, index=False, encoding="utf-8-sig")
    print(f"Combined: {out_all.name}  shape={df_all.shape}")
    display(
        df_all.groupby("location_id").agg(
            name       =("location_name", "first"),
            rows       =("datetime_utc",  "count"),
            pm25_valid =(("PM2.5 µg/m³"), lambda x: x.notna().sum()),
            start      =("datetime_utc",  "min"),
            end        =("datetime_utc",  "max"),
        )
    )
    display(df_all.head(10))
else:
    print("No data collected.")



Station 18: SPARTAN - Vietnam Acad. Sci.
  Range: 2021-01-01  →  2026-05-15


  [SKIP] No matching sensors found.

Station 2539: US Diplomatic Post: Hanoi
  Range: 2021-01-01  →  2026-05-15


  Sensors: 1
  [SKIP] No data fetched.

Station 7441: Hanoi
  Range: 2021-01-01  →  2026-05-15


  Sensors: 1
  pm25 | sensor=21632 | 112 windows
    [001/112] 2021-01-01 → 2021-01-15  

(312 records)


...

  Saved: 6123215_OceanPark.csv  shape=(4412, 10)
  PM2.5 rows: 4412


C:\Users\Admin\AppData\Local\Temp\ipykernel_28092\3028915927.py:97: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pd.concat(all_station_dfs, ignore_index=True)


Combined: all_stations_crawled.csv  shape=(137669, 10)


,name,rows,pm25_valid,start,end
location_id,,,,,
7441,Hanoi,34051,30157,2021-01-01 01:00:00+00:00,2025-04-09 15:00:00+00:00
2161290,An Khánh,9430,9416,2024-01-29 07:00:00+00:00,2025-06-10 03:00:00+00:00
2161291,Cầu Diễn,1504,1497,2024-01-22 02:00:00+00:00,2024-12-11 14:00:00+00:00
2161292,"Số 46, phố Lưu Quang Vũ",16654,16003,2024-01-29 17:00:00+00:00,2026-05-15 00:00:00+00:00
2161293,Chúc Sơn,2933,2922,2024-01-09 22:00:00+00:00,2025-02-05 08:00:00+00:00
2161294,Cung thiếu nhi,3429,3418,2024-01-29 07:00:00+00:00,2025-02-05 03:00:00+00:00
2161295,Đầm Trấu,11,11,2024-01-29 07:00:00+00:00,2024-01-30 08:00:00+00:00
2161296,Đào Duy Từ,6086,6075,2024-01-15 08:00:00+00:00,2025-06-10 03:00:00+00:00
2161298,Đông Kinh Nghĩa Thục,852,852,2024-01-29 07:00:00+00:00,2024-04-15 09:00:00+00:00


parameter_name,datetime_utc,location_id,location_name,latitude,longitude,CO mass µg/m³,NO₂ mass µg/m³,O₃ mass µg/m³,PM2.5 µg/m³,SO₂ mass µg/m³
0,2021-01-01 01:00:00+00:00,7441,Hanoi,21.021939,105.818806,NaN,NaN,NaN,70.0,NaN
1,2021-01-01 02:00:00+00:00,7441,Hanoi,21.021939,105.818806,NaN,NaN,NaN,81.0,NaN
2,2021-01-01 03:00:00+00:00,7441,Hanoi,21.021939,105.818806,NaN,NaN,NaN,49.0,NaN
3,2021-01-01 04:00:00+00:00,7441,Hanoi,21.021939,105.818806,NaN,NaN,NaN,50.0,NaN
4,2021-01-01 05:00:00+00:00,7441,Hanoi,21.021939,105.818806,NaN,NaN,NaN,50.0,NaN
5,2021-01-01 06:00:00+00:00,7441,Hanoi,21.021939,105.818806,NaN,NaN,NaN,56.0,NaN
6,2021-01-01 07:00:00+00:00,7441,Hanoi,21.021939,105.818806,NaN,NaN,NaN,46.0,NaN
7,2021-01-01 08:00:00+00:00,7441,Hanoi,21.021939,105.818806,NaN,NaN,NaN,46.0,NaN
8,2021-01-01 09:00:00+00:00,7441,Hanoi,21.021939,105.818806,NaN,NaN,NaN,46.0,NaN
9,2021-01-01 10:00:00+00:00,7441,Hanoi,21.021939,105.818806,NaN,NaN,NaN,49.0,NaN
